In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

grid           = pd.read_parquet('../data/processed/daily_sales.parquet')
model          = joblib.load('../models/hgbr_model.pkl')
build_features = joblib.load('../models/build_features.pkl')

VAL_DATES = pd.date_range('2025-08-09', '2025-09-05', freq='D')
actual    = grid.loc[VAL_DATES]
weights   = grid[grid.index >= '2025-07-12'].sum()

# ... (run all three model forecasts)

# Plot actual vs predicted — top 5 SKUs
top5 = grid.sum().sort_values(ascending=False).head(5).index
fig, axes = plt.subplots(5, 1, figsize=(14, 18))
for ax, sku in zip(axes, top5):
    ax.plot(VAL_DATES, actual[sku].values,
            label='Actual',   color='black',     linewidth=2)
    ax.plot(VAL_DATES, pred_ml[sku],
            label='ML Model', color='steelblue', linestyle='--')
    ax.plot(VAL_DATES, pred_dow[sku],
            label='DOW Avg',  color='orange',    linestyle=':')
    ax.set_title(f'{sku}  |  28d mean: {actual[sku].mean():.1f}')
    ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig('../output/eval_top5_skus.png', dpi=150)
plt.show()

# Per-tier error breakdown
for label, skus in [('Tier A', TIER_A), ('Tier B', TIER_B)]:
    mae = np.mean(np.abs(pred_final[skus].values - actual[skus].values))
    print(f"{label} MAE: {mae:.3f}")